# PYNQ 上板 MNIST 真实推理测试

本 notebook 在 PYNQ-Z2 上运行，验证 CIM SoC 能否正确识别真实 MNIST 手写数字。

**前提：**
1. 宿主机已运行 `generate_mnist_data.ipynb`，生成 `mnist_real_data.zip`
2. 已上传 `mnist_real_data.zip`、`cim_soc.bit`、`cim_soc.hwh` 到 Jupyter 同目录

**验证内容：**
1. 逐张加载真实 MNIST 测试图片
2. 硬件推理 → 逐元素对比 FC1(128) / FC2(10) / argmax 与 Python golden
3. 统计硬件准确率和 bit-exact 匹配率

In [ ]:
!unzip -o mnist_real_data.zip

In [ ]:
from pynq import Overlay, MMIO
import numpy as np
import os, glob

## 1. 加载 Bitstream

In [ ]:
ol = Overlay('cim_soc.bit')
print('Overlay loaded!')

BASE = 0x40000000
mmio = MMIO(BASE, 0x4000)  # 16KB address space

## 2. CSR 地址定义

In [ ]:
CTRL          = 0x000
STATUS        = 0x004
CSR_IN_DIM    = 0x010
CSR_OUT_DIM   = 0x014
CSR_N_IB      = 0x018
CSR_N_OB      = 0x01C
REQUANT_MULT  = 0x020
REQUANT_SHIFT = 0x024
INPUT_ZP      = 0x028
ACT_MODE      = 0x02C
CYCLE_CNT_LO  = 0x030
MAC_CNT_LO    = 0x038
PRED_CLASS    = 0x040
WDMA_ADDR     = 0x044
WDMA_DATA     = 0x048
WDMA_CTRL     = 0x04C
LOGIT_BASE    = 0x100
MEM_INPUT     = 0x1000
MEM_BIAS      = 0x2000

## 3. 硬件操作函数

In [ ]:
def soft_reset():
    mmio.write(CTRL, 0x4)

def clear_done():
    mmio.write(CTRL, 0x2)

def configure_layer(in_dim, out_dim, zp, mult, shift, act):
    n_ib = (in_dim + 15) // 16
    n_ob = (out_dim + 15) // 16
    mmio.write(CSR_IN_DIM, in_dim)
    mmio.write(CSR_OUT_DIM, out_dim)
    mmio.write(CSR_N_IB, n_ib)
    mmio.write(CSR_N_OB, n_ob)
    mmio.write(REQUANT_MULT, mult)
    mmio.write(REQUANT_SHIFT, shift)
    mmio.write(INPUT_ZP, int(zp) & 0xFFFFFFFF)
    mmio.write(ACT_MODE, act)

def load_weights_burst(chunks):
    mmio.write(WDMA_ADDR, 0)
    mmio.write(WDMA_CTRL, 0x02)
    for c in chunks:
        mmio.write(WDMA_DATA, int(c))
    mmio.write(WDMA_CTRL, 0x00)

def load_bias(bias_list):
    for i, b in enumerate(bias_list):
        mmio.write(MEM_BIAS + 4*i, int(b) & 0xFFFFFFFF)

def load_input(data_u8):
    padded = list(data_u8)
    while len(padded) % 16 != 0:
        padded.append(0)
    for i, x in enumerate(padded):
        mmio.write(MEM_INPUT + 4*i, int(x) & 0xFF)

def run_inference():
    clear_done()
    mmio.write(CTRL, 0x1)
    while not (mmio.read(STATUS) & 0x2):
        pass
    return mmio.read(CYCLE_CNT_LO), mmio.read(MAC_CNT_LO)

def read_output(out_dim):
    out = []
    for i in range(out_dim):
        v = mmio.read(LOGIT_BASE + 4*i)
        out.append(np.uint8(v & 0xFF).view(np.int8))
    return out

print('HW functions loaded.')

## 4. 读取 Hex 文件

In [ ]:
DATA_DIR = 'mnist_real_data'

def read_hex_u32(filepath):
    vals = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if line:
                vals.append(int(line, 16))
    return vals

def read_hex_u8(filepath):
    vals = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if line:
                vals.append(int(line, 16) & 0xFF)
    return vals

def read_hex_int8(filepath):
    return [np.uint8(v).view(np.int8) for v in read_hex_u8(filepath)]

## 5. 加载模型参数

In [ ]:
fc1_wchunks = read_hex_u32(f'{DATA_DIR}/fc1_weight_tiles.hex')
fc2_wchunks = read_hex_u32(f'{DATA_DIR}/fc2_weight_tiles.hex')
fc1_bias = read_hex_u32(f'{DATA_DIR}/fc1_bias.hex')
fc2_bias = read_hex_u32(f'{DATA_DIR}/fc2_bias.hex')
qp = read_hex_u32(f'{DATA_DIR}/quant_params.hex')
zp = read_hex_u32(f'{DATA_DIR}/zero_points.hex')

fc1_mult, fc1_shift = qp[0], qp[1]
fc2_mult, fc2_shift = qp[2], qp[3]
hw_zp1 = np.int32(np.uint32(zp[0]))
hw_zp2 = np.int32(np.uint32(zp[1]))

print(f'FC1 weight chunks: {len(fc1_wchunks)}')
print(f'FC2 weight chunks: {len(fc2_wchunks)}')
print(f'Quant: fc1=({fc1_mult},{fc1_shift}), fc2=({fc2_mult},{fc2_shift})')
print(f'HW zero points: FC1={hw_zp1}, FC2={hw_zp2}')

## 6. AXI 连通性测试

In [ ]:
soft_reset()
mmio.write(CSR_IN_DIM, 784)
rb = mmio.read(CSR_IN_DIM)
print(f'IN_DIM write=784, readback={rb}  {"PASS" if rb == 784 else "FAIL"}')
assert rb == 784, 'AXI connectivity failed!'

## 7. 逐张推理测试

对每张图片：加载权重/bias → 加载图片 → FC1 推理 → FC2 推理 → 对比 golden

In [ ]:
img_dir = f'{DATA_DIR}/test_images'
img_files = sorted(glob.glob(f'{img_dir}/img_????.hex'))
n_images = len(img_files)
print(f'Found {n_images} test images\n')

hw_correct = 0    # HW prediction matches true label
bit_exact = 0     # HW output bit-exact matches Python golden
total = 0
error_imgs = []

for img_path in img_files:
    name = os.path.basename(img_path).replace('.hex', '')
    
    # Read golden data
    img_u8 = read_hex_u8(img_path)
    label = int(open(f'{img_dir}/{name}_label.txt').read().strip())
    py_pred = int(open(f'{img_dir}/{name}_pred.txt').read().strip())
    fc1_golden = read_hex_int8(f'{img_dir}/{name}_fc1.hex')
    fc2_golden = read_hex_int8(f'{img_dir}/{name}_fc2.hex')
    
    # ==== FC1: 784 -> 128, ReLU ====
    soft_reset()
    configure_layer(784, 128, hw_zp1, fc1_mult, fc1_shift, 1)
    load_weights_burst(fc1_wchunks)
    load_bias(fc1_bias)
    load_input(img_u8)
    c1, m1 = run_inference()
    fc1_hw = read_output(128)
    
    # ==== FC2: 128 -> 10, no activation ====
    configure_layer(128, 10, hw_zp2, fc2_mult, fc2_shift, 0)
    load_weights_burst(fc2_wchunks)
    load_bias(fc2_bias)
    fc1_u8 = [int(x) & 0xFF for x in fc1_hw]
    load_input(fc1_u8)
    c2, m2 = run_inference()
    fc2_hw = read_output(10)
    
    hw_pred = mmio.read(PRED_CLASS)
    
    # Compare
    fc1_ok = all(fc1_hw[j] == fc1_golden[j] for j in range(128))
    fc2_ok = all(fc2_hw[j] == fc2_golden[j] for j in range(10))
    pred_ok = (hw_pred == py_pred)
    all_ok = fc1_ok and fc2_ok and pred_ok
    
    if hw_pred == label:
        hw_correct += 1
    if all_ok:
        bit_exact += 1
    else:
        error_imgs.append(name)
    total += 1
    
    mark = '\u2713' if all_ok else '\u2717'
    correct_mark = '\u2713' if hw_pred == label else '\u2717'
    print(
        f'  {name}: label={label} hw={hw_pred} py={py_pred} '
        f'fc1={"OK" if fc1_ok else "ERR"} '
        f'fc2={"OK" if fc2_ok else "ERR"} '
        f'exact={mark} correct={correct_mark}'
    )
    
    # Print details on mismatch
    if not fc1_ok:
        diffs = [j for j in range(128) if fc1_hw[j] != fc1_golden[j]]
        print(f'    FC1 diffs at indices: {diffs[:8]}...')
    if not fc2_ok:
        for j in range(10):
            if fc2_hw[j] != fc2_golden[j]:
                print(f'    FC2[{j}]: hw={fc2_hw[j]}, golden={fc2_golden[j]}')

## 8. 结果总结

In [ ]:
print('=' * 60)
print('MNIST Real Inference Results')
print('=' * 60)
print(f'  Total images tested:   {total}')
print(f'  HW accuracy (vs label): {100.*hw_correct/total:.1f}% ({hw_correct}/{total})')
print(f'  Bit-exact match (vs Python golden): {100.*bit_exact/total:.1f}% ({bit_exact}/{total})')
print(f'  Computation errors:    {len(error_imgs)}')
print()
if len(error_imgs) == 0:
    print('>>> ALL IMAGES BIT-EXACT MATCH <<<')
    print('>>> CIM SoC correctly recognizes real MNIST handwritten digits! <<<')
else:
    print(f'  Mismatched images: {error_imgs}')
print('=' * 60)